# Segmentation of CL at anode to study "crumbling"

In [2]:
import xarray as xr
import os
import socket
import numpy as np
import matplotlib.pyplot as plt
import subprocess

host = socket.gethostname()
if host == 'mpc2959.psi.ch' or host == 'mpc2053.psi.ch':
    gitpath = '/mpc/homes/fische_r/lib/co2ely-tomcat'
    toppath = '/mpc/homes/fische_r/NAS/DASCOELY'
    toppathSSD = '/mnt/SSD/fische_r/COELY'
    temppath = '/mnt/SSD/fische_r/tmp'
    training_path = '/mpc/homes/fische_r/NAS/DASCOELY/processing/05_water_GDL_ML/'
    
else:
    print('currently '+host+'not supported')
    
path_02_4D = os.path.join(toppathSSD, '02_registered_3p1D') #h5 with registered data
if not host=='mpc2959.psi.ch':
    path_02_4D = os.path.join(toppath, 'processing','02_registered_3p1D') #h5 with registered data

# fetch githash
cwd = os.getcwd()
os.chdir(gitpath)
git_sha = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
githash = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
os.chdir(cwd)

NameError: name 'subprocess' is not defined

## Data preparation
### Use ImageJ to define ROI in registered 4D-Data including Anode CL+GDL and save in dict

In [1]:
a = 332
b = a+119
print(a,b)

332 451


In [ ]:
anode_ROI = {
    '5II': (332, 451)
}

In [ ]:
th = 14500
thmax = 3*th #to exlude registration artifacts

## Sample processing

### load ROI of selected sample

In [ ]:
sample = '5II

file = '02_'+sample+'_registered_3p1D.nc'
imagepath = os.path.join(path_02_4D, file)
data = xr.open_dataset(imagepath)
images = [im for im in data.keys() if im[3:7] == 'imag']
images.sort()

In [ ]:
a,b = anode_ROI[sample]
shp = data[images[4]][a:b, :, :].shape
shp = shp + (len(images),)

In [ ]:
im = np.zeros(shp, dtype = np.uint16)

In [ ]:
for i in range(shp[-1]):
    if i%10==0:
        print(i)
    im[...,i] = data[images[i]][a:b, c:d, e:f].data

### segment data

In [ ]:
segs = np.bitwise_and(im>th, im<thmax)

### save to netcdf4 to be fed in Blender pipeline

In [ ]:
outpath = os.path.join(toppath, 'processing', , '06_anode_CL'+sample'_anode_CL_segmented.nc')
shp = segs.shape
segdata = xr.Dataset({'segmented': (['x','y','z','time'], segs)},
                               coords = {'x': np.arange(shp[0]),
                               'y': np.arange(shp[1]),
                               'z': np.arange(shp[2]),
                               'time': np.arange(shp[3]),
                               'feature': SM.feature_names},
                     attrs = {'name': sample,
                             'cropping': [a,b,c,d,e,f]}
                     )